In [1]:
!pip install face_recognition


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566242 sha256=8d2e318f66f4c92cfd46dfc3c6de8b77ba6756f5baf877567ac4d632d948ad63
  Stored in directory: /root/.cache/pip/wheels/39/20/47/6b647a3457f0e36c161b2516a5bd7f29f6bbd5e3a67c46a710
Successfully built face-recognition-models


In [4]:
import os
import datetime
import numpy as np
import pandas as pd
from PIL import Image
import face_recognition
from IPython.display import display, Javascript
from google.colab.output import eval_js
from json import JSONEncoder

# 1. THE WEBCAM ENGINE (JavaScript Bridge for Google Colab)
def take_photo(filename='photo.jpg', quality=0.8):
  """Uses JavaScript to access the laptop webcam from inside the browser."""
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture Frame';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture button to be clicked
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))

  # Decode base64 image data string into an actual image file
  import base64
  binary = base64.b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# 2. VIRTUAL DATABASE STORAGE

# In-memory dictionaries to hold student records for this notebook session
registered_students = {}  # Format: {'Student Name': [128-facial-embeddings]}
attendance_log = []       # Format: [{'Name': 'Name', 'Timestamp': 'Time'}]

# =====================================================================
# 3. STUDENT REGISTRATION FUNCTION
def register_new_student():
    """Prompts for a name, turns on webcam, and registers the student."""
    name = input("Enter the student's Full Name to register: ").strip()
    if not name:
        print(" Name cannot be empty.")
        return

    print(f"\n Get ready, {name}! Look at the webcam and click 'Capture Frame'...")
    try:
        # Take photo and save as temporary file
        img_path = take_photo('temp_register.jpg')

        # Process the image to look for facial signatures
        image = face_recognition.load_image_file(img_path)
        face_encodings = face_recognition.face_encodings(image)

        if len(face_encodings) > 0:
            # Save the first face found in the frame
            registered_students[name] = face_encodings[0]
            print(f"✅ Successfully registered: {name}!")
        else:
            print(" No face detected. Please try registering again in better lighting.")

        # Clean up temp file
        if os.path.exists(img_path): os.remove(img_path)

    except Exception as e:
        print(f"An error occurred during webcam access: {e}")


# 4. ATTENDANCE DETECTION FUNCTION
def scan_attendance():
    """Scans webcam feed, detects student, updates logs, and shows a pop-up."""
    if not registered_students:
        display(Javascript('alert("Error: No students registered in the database yet!");'))
        print("Cannot scan. The database is empty...")
        return

    print("\nScanning... Look at the webcam and click 'Capture Frame'...")
    try:
        img_path = take_photo('temp_attendance.jpg')

        image = face_recognition.load_image_file(img_path)
        current_encodings = face_recognition.face_encodings(image)

        if len(current_encodings) == 0:
            display(Javascript('alert("No face detected by the camera. Try again.");'))
            return

        # Extract features of the scanned face
        scanned_face = current_encodings[0]

        # Prepare known lists for comparison
        known_names = list(registered_students.keys())
        known_encodings = list(registered_students.values())

        # Compare scanned face against all registered faces (tolerance 0.6 is industry standard)
        matches = face_recognition.compare_faces(known_encodings, scanned_face, tolerance=0.6)

        if True in matches:
            # Find the index of the matching true value
            match_index = matches.index(True)
            matched_name = known_names[match_index]

            # Log the attendance records
            timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            attendance_log.append({"Name": matched_name, "Timestamp": timestamp})
            print(f"Logged: {matched_name} at {timestamp}")

            # TRIGGER THE BROWSER POP-UP
            display(Javascript(f'alert("Attendance Recorded: {matched_name}");'))
        else:
            # Face did not match anyone in the system
            display(Javascript('alert("Access Denied: Face Not Recognized.");'))

        if os.path.exists(img_path): os.remove(img_path)

    except Exception as e:
        print(f"An error occurred during webcam access: {e}")

# 5. VIEW ATTENDANCE REPORT
def print_attendance_sheet():
    """Prints out the current logged attendance sheet as a clean table."""
    if not attendance_log:
        print("\nAttendance Sheet is currently empty.")
    else:
        df = pd.DataFrame(attendance_log)
        print("\nCURRENT ATTENDANCE SHEET:")
        display(df)


In [6]:
register_new_student()


Enter the student's Full Name to register: Mahesh

📸 Get ready, Mahesh! Look at the webcam and click 'Capture Frame'...


<IPython.core.display.Javascript object>

✅ Successfully registered: Mahesh!


In [7]:
scan_attendance()



🔍 Scanning... Look at the webcam and click 'Capture Frame'...


<IPython.core.display.Javascript object>

📝 Logged: Mahesh at 2026-10-20 11:53:57


<IPython.core.display.Javascript object>

In [8]:
print_attendance_sheet()



📊 CURRENT ATTENDANCE SHEET:


,Name,Timestamp
0,Mahesh,2026-10-20 11:53:57
